# Lab 1 · ผ่าโมเดล: อ่านพิมพ์เขียวและนับพารามิเตอร์ของ Typhoon2-1B
**สัปดาห์ 3 · ช่วงเช้า (สาธิตสด 30 นาที)** — ผู้สอนรันบนจอ นักศึกษาเปิดตามได้

สิ่งที่จะได้เห็นใน notebook นี้
1. อ่าน `config.json` ของ Typhoon2-1B และ Qwen3-1.7B แล้วเทียบตัวเลข
2. tokenizer สองตัวตัดประโยคไทยเดียวกันต่างกันอย่างไร
3. โครงสร้างจริงของโมเดล: `embed_tokens → layers[0..15] → norm → lm_head`
4. นับพารามิเตอร์ด้วยโค้ด แล้วเทียบกับที่คำนวณด้วยมือในสไลด์ (≈ 1.24B)
5. ส่ง tensor ผ่านทีละชิ้นแล้วดู shape

> **ก่อนรัน:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# 0) ติดตั้งและ import  (Qwen3 ต้องใช้ transformers 4.51 ขึ้นไป)
!pip install -q -U "transformers>=4.51,<5" accelerate
import torch, transformers
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
import warnings, transformers; warnings.filterwarnings("ignore"); transformers.utils.logging.set_verbosity_error()   # ซ่อน warning ที่ไม่เกี่ยวกับบทเรียน
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่พบ GPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.57.6 | GPU: Tesla T4


In [2]:
TYPHOON = "scb10x/llama3.2-typhoon2-1b-instruct"
QWEN    = "Qwen/Qwen3-1.7B"

## 1.1 อ่านพิมพ์เขียว (config) ของทั้งสองโมเดล — ไม่ต้องโหลด weight
`AutoConfig` อ่านแค่ไฟล์ `config.json` จึงเร็วและไม่กินหน่วยความจำ

In [3]:
cfg_t = AutoConfig.from_pretrained(TYPHOON)
cfg_q = AutoConfig.from_pretrained(QWEN)

fields = ["architectures", "vocab_size", "hidden_size", "num_hidden_layers", "num_attention_heads",
          "num_key_value_heads", "intermediate_size", "max_position_embeddings", "tie_word_embeddings", "hidden_act"]
print(f"{'ค่าใน config':28s} {'Typhoon2-1B':>18s} {'Qwen3-1.7B':>18s}")
for f in fields:
    a, b = getattr(cfg_t, f, None), getattr(cfg_q, f, None)
    a, b = (a[0] if isinstance(a, list) else a), (b[0] if isinstance(b, list) else b)
    print(f"{f:28s} {str(a):>18s} {str(b):>18s}")

# head_dim = hidden_size / num_attention_heads (Llama) — Qwen3 ระบุ head_dim แยกไว้
print(f"{'head_dim':28s} {cfg_t.hidden_size // cfg_t.num_attention_heads:>18d} {getattr(cfg_q, 'head_dim', cfg_q.hidden_size // cfg_q.num_attention_heads):>18d}")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

ค่าใน config                        Typhoon2-1B         Qwen3-1.7B
architectures                  LlamaForCausalLM   Qwen3ForCausalLM
vocab_size                               128256             151936
hidden_size                                2048               2048
num_hidden_layers                            16                 28
num_attention_heads                          32                 16
num_key_value_heads                           8                  8
intermediate_size                          8192               6144
max_position_embeddings                  131072              40960
tie_word_embeddings                        True               True
hidden_act                                 silu               silu
head_dim                                     64                128


**อ่านตารางนี้อย่างไร** (โยงกับสไลด์ "อ่าน config")
- `hidden_size` เท่ากัน (2,048) แต่ Qwen3 มีชั้นมากกว่า (28 vs 16) และ feed-forward แคบกว่า (6,144 vs 8,192)
- `num_key_value_heads` = 8 ทั้งคู่ แต่ Typhoon2 มี 32 หัว query (GQA แบบ 4 ต่อ 1) ส่วน Qwen3 มี 16 หัว (2 ต่อ 1)
- `tie_word_embeddings = True` ทั้งคู่ → output head ใช้ตาราง embedding ร่วม ไม่นับพารามิเตอร์ซ้ำ

## 1.2 tokenizer สองตัว ตัดประโยคไทยเดียวกันต่างกันอย่างไร
(ต่อจาก Lab 1.2 สัปดาห์ 2 ที่พบว่าภาษาไทยใช้ token มากกว่าภาษาอังกฤษ)

In [4]:
tok_t = AutoTokenizer.from_pretrained(TYPHOON)
tok_q = AutoTokenizer.from_pretrained(QWEN)

sentences = [
    "เมืองหลวงของประเทศไทยคือกรุงเทพมหานคร",
    "หูฟังไร้สาย SoundPro X2 ราคา 1,290 บาท ส่งฟรี",
    "The capital of Thailand is Bangkok.",
]
def pieces(tok, s):
    # ถอดแต่ละ token กลับเป็นข้อความ เพื่อให้อ่านออก (tokenizer เก็บอักษรไทยเป็นไบต์ ถ้าพิมพ์ดิบจะเห็นเป็น à¸...)
    # token ที่เป็น "เศษไบต์" ของอักษรไทย (ไม่ครบตัวอักษร) จะแสดงเป็น �
    return [tok.decode([i]) for i in tok(s, add_special_tokens=False).input_ids]

for s in sentences:
    a, b = pieces(tok_t, s), pieces(tok_q, s)
    print("ประโยค:", s)
    print(f"  Typhoon2 ({len(a):2d} token): " + " | ".join(a))
    print(f"  Qwen3    ({len(b):2d} token): " + " | ".join(b))
    print()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

ประโยค: เมืองหลวงของประเทศไทยคือกรุงเทพมหานคร
  Typhoon2 ( 9 token): เม | ือง | หลวง | ของ | ประเทศไทย | ค | ือ | กร | ุงเทพมหานคร
  Qwen3    (14 token): เม | ือง | หลวง | ของ | ประเทศไทย | ค | ือ | กร | ุง | เทพ | ม | ห | าน | คร

ประโยค: หูฟังไร้สาย SoundPro X2 ราคา 1,290 บาท ส่งฟรี
  Typhoon2 (21 token): ห | ู | ฟ | ัง | ไร | ้ส | าย |  Sound | Pro |  X | 2 |  ราคา |   | 1 | , | 290 |  บาท |  ส | ่ง | ฟร | ี
  Qwen3    (25 token): ห | ู | ฟ | ัง | ไร | ้ | สาย |  Sound | Pro |  X | 2 |   | ราคา |   | 1 | , | 2 | 9 | 0 |  บาท |  ส | ่ง | ฟ | ร | ี

ประโยค: The capital of Thailand is Bangkok.
  Typhoon2 ( 7 token): The |  capital |  of |  Thailand |  is |  Bangkok | .
  Qwen3    ( 7 token): The |  capital |  of |  Thailand |  is |  Bangkok | .



> แต่ละชิ้นที่คั่นด้วย `|` คือ 1 token · `�` = token ที่เป็นเศษไบต์ของอักษรไทย ยังไม่ครบหนึ่งตัวอักษร (byte-level BPE) — คำไทยหนึ่งคำอาจถูกตัดเป็นหลาย token
> ประโยคอังกฤษมักได้ 1 คำ ≈ 1 token ส่วนภาษาไทยใช้ token มากกว่า → มีผลตรงต่อค่าใช้จ่ายและความยาว context ที่ใช้ได้

## 1.3 โหลด weight ของ Typhoon2 (ตัวเดียว, float16) แล้วดูโครงสร้างจริง
โหลดแค่ตัวเดียวเพื่อไม่ให้เกินหน่วยความจำของ Colab ฟรี · float16 ใช้ RAM ประมาณครึ่งหนึ่งของ float32

In [5]:
model = AutoModelForCausalLM.from_pretrained(TYPHOON, torch_dtype=torch.float16, device_map="auto")
model.eval()
print(model)

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

**สิ่งที่ควรชี้บนจอ** — ไล่จากบนลงล่างตรงกับสไลด์ "ประกอบโมเดลทีละชิ้น"
- `embed_tokens: Embedding(128256, 2048)` → ชิ้น ①
- `layers: ModuleList(0-15)` แต่ละชั้นมี `self_attn` (q/k/v/o_proj) และ `mlp` (gate/up/down_proj) คั่นด้วย `input_layernorm` / `post_attention_layernorm` → ชิ้น ② ซ้อน 16 ชั้น (③)
- `norm` → normalization ครั้งสุดท้าย · `rotary_emb` = RoPE ที่ใส่ข้อมูลตำแหน่งให้ Q/K (ไม่มีพารามิเตอร์ให้เทรน) · `act_fn: SiLU` = ฟังก์ชันใน SwiGLU
- `lm_head: Linear(2048 → 128256)` → ชิ้น ④ (weight ผูกกับ embed_tokens)
- สังเกต `k_proj: Linear(2048 → 512)` เล็กกว่า `q_proj: Linear(2048 → 2048)` — นี่คือ GQA

## 1.4 นับพารามิเตอร์ด้วยโค้ด แล้วเทียบกับที่คำนวณด้วยมือ

In [6]:
from collections import defaultdict
buckets = defaultdict(int)
for name, p in model.named_parameters():          # weight ที่ผูกกัน (tie) จะถูกนับครั้งเดียว
    if "embed_tokens" in name:  buckets["embedding"] += p.numel()
    elif "self_attn" in name:   buckets["attention (16 ชั้น)"] += p.numel()
    elif "mlp" in name:         buckets["feed-forward (16 ชั้น)"] += p.numel()
    elif "norm" in name:        buckets["norm ทั้งหมด"] += p.numel()
    else:                       buckets["อื่น ๆ: " + name] += p.numel()
total = sum(buckets.values())
for k, v in buckets.items():
    print(f"{k:26s} {v/1e6:9.1f} ล้าน  ({v/total*100:4.1f}%)")
print(f"{'รวม':26s} {total/1e6:9.1f} ล้าน  ≈ {total/1e9:.2f}B")

# ---- คำนวณด้วยมือจาก config (สูตรเดียวกับสไลด์) ----
H, L, V, I = cfg_t.hidden_size, cfg_t.num_hidden_layers, cfg_t.vocab_size, cfg_t.intermediate_size
kv = cfg_t.num_key_value_heads * (H // cfg_t.num_attention_heads)     # 8 หัว × 64 = 512
emb  = V * H
attn = H*H + H*kv + H*kv + H*H            # Q, K, V, O
ffn  = 3 * H * I                          # gate, up, down (SwiGLU)
hand = emb + L * (attn + ffn)
print(f"\nคำนวณด้วยมือ: embedding {emb/1e6:.1f} + 16 × (attention {attn/1e6:.1f} + ffn {ffn/1e6:.1f}) = {hand/1e6:.1f} ล้าน")
print(f"ต่างจากของจริง {abs(total-hand):,} ตัว (= norm ที่เราตัดทิ้งตอนคำนวณมือ)")

embedding                      262.7 ล้าน  (21.3%)
attention (16 ชั้น)            167.8 ล้าน  (13.6%)
feed-forward (16 ชั้น)         805.3 ล้าน  (65.2%)
norm ทั้งหมด                     0.1 ล้าน  ( 0.0%)
รวม                           1235.8 ล้าน  ≈ 1.24B

คำนวณด้วยมือ: embedding 262.7 + 16 × (attention 10.5 + ffn 50.3) = 1235.7 ล้าน
ต่างจากของจริง 67,584 ตัว (= norm ที่เราตัดทิ้งตอนคำนวณมือ)


## 1.5 ส่ง tensor ผ่านทีละชิ้น แล้วดู shape
ใช้ *forward hook* ดักดู output ของแต่ละชิ้น: shape จะเป็น `(1, N, 2048)` คงที่ตลอด 16 ชั้น และเปลี่ยนเป็น `(1, N, 128256)` เฉพาะที่ `lm_head`

In [7]:
text = "เมืองหลวงของประเทศไทยคือ"
enc = tok_t(text, return_tensors="pt").to(model.device)
print("token ids shape:", tuple(enc.input_ids.shape), "→", " | ".join(tok_t.decode([i]) for i in enc.input_ids[0]), "  (token แรกคือ <|begin_of_text|>)")

def hook(name):
    def _h(module, inp, out):
        o = out[0] if isinstance(out, tuple) else out
        print(f"{name:22s} output shape: {tuple(o.shape)}")
    return _h

handles = [
    model.model.embed_tokens.register_forward_hook(hook("① embed_tokens")),
    model.model.layers[0].register_forward_hook(hook("② layers[0]")),
    model.model.layers[15].register_forward_hook(hook("② layers[15]")),
    model.model.norm.register_forward_hook(hook("   norm")),
    model.lm_head.register_forward_hook(hook("④ lm_head")),
]
with torch.no_grad():
    out = model(**enc)
for h in handles: h.remove()

# ความน่าจะเป็นของ token ถัดไป จากตำแหน่งสุดท้าย
probs = torch.softmax(out.logits[0, -1].float(), dim=-1)
top = torch.topk(probs, 5)
print("\ntoken ถัดไป 5 อันดับ:")
for p, i in zip(top.values, top.indices):
    print(f"  {p.item():.3f}  {tok_t.decode([i.item()])!r}")

token ids shape: (1, 8) → <|begin_of_text|> | เม | ือง | หลวง | ของ | ประเทศไทย | ค | ือ   (token แรกคือ <|begin_of_text|>)
① embed_tokens         output shape: (1, 8, 2048)
② layers[0]            output shape: (1, 8, 2048)
② layers[15]           output shape: (1, 8, 2048)
   norm                output shape: (1, 8, 2048)
④ lm_head              output shape: (1, 8, 128256)

token ถัดไป 5 อันดับ:
  0.733  'กร'
  0.059  'อะไร'
  0.059  ' กร'
  0.030  'เม'
  0.024  'จ'


> **สิ่งที่เห็น:** shape `(1, N, 2048)` ไม่เปลี่ยนเลยตั้งแต่ embedding จนถึง norm — ทุกชั้น "แก้ไข" vector เดิม ไม่ได้สร้างของใหม่ · มาเปลี่ยนเป็น `(1, N, 128256)` ที่ `lm_head` เพราะต้องให้คะแนนทุก token ใน vocab · top-5 ด้านล่างคือ token ถัดไปที่โมเดลคิดว่าน่าจะมาต่อจาก "เมืองหลวงของประเทศไทยคือ"

## 1.6 (เสริม) หน่วยความจำของ KV cache — ทำไม GQA ถึงสำคัญ
ตอน generate โมเดลต้องจำ key/value ของทุก token ที่ผ่านมา · ลองคำนวณว่า context เต็ม 131,072 token ใช้กี่ GB

In [8]:
head_dim = H // cfg_t.num_attention_heads
bytes_per_token = 2 * L * cfg_t.num_key_value_heads * head_dim * 2      # (K และ V) × ชั้น × หัว KV × มิติ × 2 ไบต์ (fp16)
bytes_no_gqa    = 2 * L * cfg_t.num_attention_heads  * head_dim * 2      # ถ้าไม่มี GQA (หัว KV = หัว Q = 32)
ctx = cfg_t.max_position_embeddings
print(f"KV cache ต่อ token: {bytes_per_token/1024:.0f} KB  → context {ctx:,} token ≈ {bytes_per_token*ctx/1e9:.1f} GB")
print(f"ถ้าไม่มี GQA:      {bytes_no_gqa/1024:.0f} KB  → ≈ {bytes_no_gqa*ctx/1e9:.1f} GB  (GPU T4 มี ≈ 15 GB)")

KV cache ต่อ token: 32 KB  → context 131,072 token ≈ 4.3 GB
ถ้าไม่มี GQA:      128 KB  → ≈ 17.2 GB  (GPU T4 มี ≈ 15 GB)


## ✅ สรุป Lab 1
- config คือพิมพ์เขียว: อ่านแล้วรู้ทันทีว่าโมเดลกว้าง/ลึก/รู้จักกี่ token
- Typhoon2-1B = โครงสร้าง Llama 3.2 ทุกค่า (สิ่งที่เปลี่ยนคือ weight) · Qwen3 เป็นโครง decoder แบบเดียวกัน (RMSNorm + GQA + SwiGLU + RoPE) ต่างที่ตัวเลข และเพิ่ม QK-Norm ใน attention
- นับพารามิเตอร์ด้วยมือได้ตรงกับของจริง: feed-forward กินมากที่สุด (≈ 65%) embedding ≈ 21%
- shape คงที่ (1 × N × 2,048) ตลอดทาง เปลี่ยนเป็นขนาด vocab ที่ output head เท่านั้น